# CTA holidays and their effect on bus ridership

Which days CTA actually runs a holiday schedule on, and how much ridership changes when it does.
Split out of `exploration.ipynb`, which this notebook depends on.

**Run `exploration.ipynb` first** — it writes `data/derived/daily.csv`, read below.

### Outline
0. Setup: load the cleaned daily data
1. Detecting CTA's operational holidays, and naming them
2. What a holiday does to system-wide ridership
3. Whether the effect depends on the route

This notebook writes `data/derived/holiday_calendar.csv`, which `seasonality.ipynb` needs.

## 0. Setup: load the cleaned daily data

Imports and plotting parameters are the same block as in `exploration.ipynb`.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D

# Okabe-Ito: the published colour-vision-deficiency-safe categorical set.
BLUE, ORANGE, GREEN, PURPLE = '#0072B2', '#D55E00', '#009E73', '#CC79A7'
GRAY, INK = '#C9C9C9', '#333333'

# Ridership regimes. These are a COLOUR SCHEME for reading charts over time, not an
# analysis grouping: any analysis that needs a particular window defines it locally and
# prints what it used. 2020 gets its own colour because it is not comparable to anything
# else; the recovery years are lighter shades of it because the system has not returned
# to the pre-2020 level; 2025-present is distinct because the Frequent Network rollout
# begins 2025-03-23, so those years are not a clean baseline for anything.
ERAS = [('pre-2020',     2001, 2019, BLUE),
        ('2020',         2020, 2020, ORANGE),
        ('2021-2022',    2021, 2022, '#EE8A4E'),
        ('2023-2024',    2023, 2024, '#F5BE99'),
        ('2025-present', 2025, 2026, GREEN)]
ERA_ORDER = [e[0] for e in ERAS]
ERA_COLOR = {e[0]: e[3] for e in ERAS}

def era(year):
    """Map a calendar year to its ridership regime."""
    for name, lo, hi, _ in ERAS:
        if lo <= year <= hi:
            return name
    return None

# The 20 Frequent Network routes, as CTA labels them. Defined here because several
# sections need it, including the corridor check in section 2.
FREQ = ['J14', '4', '9', '12', '20', '34', '47', '49', '53', '54',
        '55', '60', '63', '66', '72', '77', '79', '81', '82', '95']

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#9A9A9A', 'axes.grid': True,
    'grid.color': '#E8E8E8', 'grid.linewidth': 0.8,
})
fmt_riders = FuncFormatter(lambda v, _: f'{v*1e-6:.1f}M' if v >= 1e6 else f'{v*1e-3:.0f}k')

In [ ]:
d = pd.read_csv('data/derived/daily.csv',
                dtype={'route': str, 'corridor': str},
                parse_dates=['date', 'week'])
print(f'rows {len(d):,}   routes {d.route.nunique()}   '
      f'{d.date.min().date()} .. {d.date.max().date()}')
print(f'columns: {list(d.columns)}')

# System-wide daily totals, same construction as in exploration.ipynb section 5.
day = d.groupby('date', as_index=False).rides.sum()
day['dow']  = day.date.dt.dayofweek
day['year'] = day.date.dt.year
day['era']  = day.year.map(era)
print(f'\nsystem daily totals: {len(day):,} days')

## 1. Detecting CTA's operational holidays

`daytype` is CTA's own label for which schedule ran: `W` weekday, `A` Saturday, `U` Sunday.
The true day of week comes from `date`. **A date whose label disagrees with its actual day of
week is a CTA-designated holiday** — a Monday run on a Sunday schedule.

Detecting them this way gives CTA's *operational* holiday list, which is the one that matters
for ridership. Names then come from `pandas`' `USFederalHolidayCalendar`, which also tells us
which federal holidays CTA does **not** treat as holidays.

In [ ]:
# Do all routes agree on the daytype for a given date?
lab = d.groupby('date').daytype.agg(n='nunique', label='first')
print(f'dates where routes disagree on daytype : {int((lab.n > 1).sum()):,}')

cal = lab.reset_index()[['date', 'label']]
cal['dow']      = cal.date.dt.dayofweek
cal['expected'] = np.where(cal.dow <= 4, 'W', np.where(cal.dow == 5, 'A', 'U'))
cal['holiday']  = cal.label != cal.expected

print(f'dates total                            : {len(cal):,}')
print(f'dates flagged as holiday               : {int(cal.holiday.sum()):,}')
print(f'  weekday run on a Sunday schedule     : '
      f'{int(((cal.dow <= 4) & (cal.label == "U")).sum()):,}')
print(f'  weekday run on a Saturday schedule   : '
      f'{int(((cal.dow <= 4) & (cal.label == "A")).sum()):,}')
print(f'  weekend date NOT on its own schedule : '
      f'{int(((cal.dow >= 5) & cal.holiday).sum()):,}')

h = cal[cal.holiday].copy()

# Weekend dates that did not run their own schedule -- odd enough to look at individually.
odd = h[h.dow >= 5]
print(f'\nweekend dates on an unexpected schedule ({len(odd)}):')
print(odd[['date', 'label', 'expected']].to_string(index=False))

In [ ]:
from pandas.tseries.holiday import USFederalHolidayCalendar

# Look the names up rather than inferring them from calendar position. The calendar
# already applies the federal "observed" shift, which is what CTA follows too.
fed = USFederalHolidayCalendar().holidays(d.date.min(), d.date.max(), return_name=True)

h['name'] = h.date.map(fed)
print(f'matched on the federal OBSERVED date : {int(h.name.notna().sum())} of {len(h)}')

# The calendar lists the observed date, which shifts to the nearest weekday when a
# fixed-date holiday lands on a weekend. CTA sometimes runs the holiday schedule on the
# true date instead, so fall back to matching the actual month/day.
FIXED_DATE = {(1, 1):  "New Year's Day",
              (6, 19): 'Juneteenth National Independence Day',
              (7, 4):  'Independence Day',
              (11, 11): 'Veterans Day',
              (12, 25): 'Christmas Day'}

miss = h.name.isna()
if miss.any():
    h.loc[miss, 'name'] = [FIXED_DATE.get((t.month, t.day)) for t in h.loc[miss, 'date']]
    print(f'matched on the true (unshifted) date  : {int(miss.sum() - h.name.isna().sum())}')
    print(h.loc[miss, ['date', 'label', 'expected', 'name']].to_string(index=False))

still = h.name.isna()
print(f'\nstill unnamed: {int(still.sum())}')
if still.any():
    print(h.loc[still, ['date', 'label', 'expected']].to_string(index=False))
    h.loc[still, 'name'] = h.loc[still, 'date'].dt.strftime('%b %d') + ' (unnamed)'

print('\nflagged dates by name:')
print(h.name.value_counts().rename('dates').to_frame().to_string())

Which holidays count is not fixed in time — Juneteenth only became a federal holiday in 2021,
and `USFederalHolidayCalendar` reflects that. So the comparison below is **by year**, not a
single set difference, which would have hidden the change.

Only weekday occurrences are detectable: a holiday falling on a weekend already runs the
Saturday or Sunday schedule, so there is no label to disagree with.

In [ ]:
fed_wd = (fed.rename('name').rename_axis('date').reset_index()
             .assign(year=lambda t: t.date.dt.year)
             .query('date.dt.dayofweek <= 4'))                 # weekday occurrences only

flagged = set(zip(h.name, h.date.dt.year))
fed_wd['cta'] = [(n, y) in flagged for n, y in zip(fed_wd.name, fed_wd.year)]

# Both columns count DATES, not years: 2021 holds two federal New Year's Days
# (Jan 1, and Dec 31 as the observance of 2022's), so a year count would not line up.
print((fed_wd.groupby('name')
             .agg(federal_weekday_dates=('cta', 'size'),
                  first=('year', 'min'), last=('year', 'max'),
                  cta_ran_holiday_schedule=('cta', 'sum'))
             .sort_values('cta_ran_holiday_schedule', ascending=False)
             .to_string()))

### Save the holiday calendar

`seasonality.ipynb` holds these dates out of the seasonal profile, so it reads this file.

In [ ]:
from pathlib import Path

DERIVED = Path('data/derived')
DERIVED.mkdir(parents=True, exist_ok=True)

hol_path = DERIVED / 'holiday_calendar.csv'
out = cal.merge(h[['date', 'name']], on='date', how='left')
out.to_csv(hol_path, index=False)
print(f'{str(hol_path):<34} {len(out):>10,} rows '
      f'({int(out.holiday.sum()):,} flagged as holidays)')
print(f'  columns: {list(out.columns)}')

## 2. What a holiday does to system-wide ridership

In [ ]:
# ---------------------------------------------------------------------------
# PROPOSED definition of "usual", flagged for review before anything is built on it:
#   usual(date) = median system ridership on the SAME day of week within +/- 4 weeks,
#                 excluding dates that are themselves flagged holidays.
# ---------------------------------------------------------------------------
sysday = day.set_index('date').rides
hol_dates = set(cal.loc[cal.holiday, 'date'])

def usual(dt, dow, span_days=28):
    win = sysday[(sysday.index >= dt - pd.Timedelta(days=span_days)) &
                 (sysday.index <= dt + pd.Timedelta(days=span_days))]
    win = win[(win.index.dayofweek == dow) & (~win.index.isin(hol_dates))]
    return (win.median(), len(win))

rows = []
for dt, dw, nm in zip(h.date, h.dow, h.name):
    u, n = usual(dt, dw)
    rows.append({'date': dt, 'name': nm, 'actual': sysday.get(dt, np.nan),
                 'usual': u, 'n_baseline': n})
hr = pd.DataFrame(rows)
hr['ratio'] = hr.actual / hr.usual

print(f'holiday dates scored          : {len(hr):,}')
print(f'  with no usable baseline     : {int(hr.usual.isna().sum()):,}')
print(f'  baseline dates used, median : {hr.n_baseline.median():.0f} (of 8 possible)')
print(f'  baseline built from < 4 days: {int((hr.n_baseline < 4).sum()):,}')

In [ ]:
def box_with_points(ax, groups, labels, colour, ylabel, title, logy=False, names=None,
                    max_labels=3):
    """One box per group, individual observations jittered on top.

    If ``names`` is given (one array of labels per group), points outside the
    1.5*IQR fences are annotated, at most ``max_labels`` per side per group.
    Returns the outliers as a DataFrame so the full list can be printed.
    """
    ax.axhline(1, color=INK, lw=0.9, ls=':')
    bp = ax.boxplot(groups, positions=range(len(groups)), widths=0.6, showfliers=False,
                    patch_artist=True, medianprops={'color': ORANGE, 'lw': 1.8})
    for box in bp['boxes']:
        box.set(facecolor='#EFEFEF', edgecolor='#9A9A9A', lw=0.8)

    found = []
    for i, g in enumerate(groups):
        ax.scatter(i + np.random.uniform(-0.14, 0.14, len(g)), g,
                   s=10, color=colour, alpha=0.5, linewidths=0, zorder=3)
        if names is None:
            continue
        q1, q3 = np.percentile(g, [25, 75])
        fence = 1.5 * (q3 - q1)
        out = [(v, n) for v, n in zip(g, names[i]) if v < q1 - fence or v > q3 + fence]
        for v, n in out:
            found.append({'group': labels[i].split(chr(10))[0], 'name': n, 'value': v})
        low  = sorted([o for o in out if o[0] < q1], key=lambda t: t[0])[:max_labels]
        high = sorted([o for o in out if o[0] > q3], key=lambda t: -t[0])[:max_labels]
        for k, (v, n) in enumerate(low + high):
            ax.annotate(n, (i, v), xytext=(7, 6 if k % 2 else -6),
                        textcoords='offset points',
                        fontsize=7, color=INK, va='center')

    if logy:
        ax.set_yscale('log')
    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc='left', fontsize=11)
    return pd.DataFrame(found)

In [ ]:
order  = hr.dropna(subset=['ratio']).groupby('name').ratio.median().sort_values().index
groups = [hr.loc[hr.name == nm, 'ratio'].dropna().to_numpy() for nm in order]

fig, ax = plt.subplots(figsize=(9.5, 4.2))
box_with_points(ax, groups, [f'{nm}\n(n={len(g)})' for nm, g in zip(order, groups)], BLUE,
                'system rides ÷ usual for that weekday',
                'Holiday ridership vs the same weekday nearby — each point is one year')
plt.show()

print(hr.groupby('name').ratio.agg(['count', 'median', 'min', 'max']).round(3).to_string())

## 3. Does the holiday effect depend on the route?

Same ratio, computed per route instead of system-wide, pooled across all years. A route whose
box sits well above the system median is one where the holiday matters less — or where the
holiday itself generates trips.

In [ ]:
rt = d.pivot_table(index='date', columns='route', values='rides', aggfunc='sum')

ratios, base_size = [], []
for dt, dw, nm in zip(h.date, h.dow, h.name):
    win = rt[(rt.index.dayofweek == dw) & ~rt.index.isin(hol_dates) &
             (rt.index >= dt - pd.Timedelta(days=28)) & (rt.index <= dt + pd.Timedelta(days=28))]
    base = win.median()
    ratios.append((rt.loc[dt] / base).replace([np.inf, -np.inf], np.nan).rename((nm, dt)))
    base_size.append(base.rename((nm, dt)))

pr   = pd.DataFrame(ratios)                       # (holiday, date) x route
base = pd.DataFrame(base_size)
print(f'route x holiday ratios computed : {int(pr.notna().sum().sum()):,}')
print(f'  ratio > 3 (route grew 3x+)    : {int((pr > 3).sum().sum()):,}')
print(f'  of those, usual < 100 rides/day: {int(((pr > 3) & (base < 100)).sum().sum()):,}')
print('\nExtreme ratios come from routes with a tiny baseline, so nothing is dropped —')
print('the axis below is log-scaled instead, and every route is shown.')

In [ ]:
med  = pr.groupby(level=0).median()               # holiday x route, median across years
keep = [nm for nm in order if nm in med.index]
groups = [med.loc[nm].dropna() for nm in keep]

fig, ax = plt.subplots(figsize=(10.5, 4.6))
outliers = box_with_points(
    ax, [g.to_numpy() for g in groups],
    [f'{nm}\n({len(g)} routes)' for nm, g in zip(keep, groups)], GREEN,
    "route rides ÷ that route's usual weekday",
    'Holiday effect by route — each point is one route, median across years',
    logy=True, names=[g.index.to_numpy() for g in groups])
plt.show()

print(f'routes outside the 1.5xIQR fences: {len(outliers)} '
      f'(plot labels at most 4 per side per holiday)')
with pd.option_context('display.max_rows', None):
    display(outliers.sort_values(['group', 'value']).reset_index(drop=True).round(3))

## Where this leaves us

### Established

- CTA runs a holiday schedule on exactly six holidays, never on the other five federal ones,
  and the effect is large — 0.26x to 0.53x of a normal weekday, varying a lot by route (§2, §3).

### Decisions still open

1. **The `usual` definition in §2** (median of the same weekday within ±4 weeks) is a proposal,
   flagged inline where it is defined.